# Unit 6 — Final Assessment · Part A
## Improving the YOLO26 Cat Detector, then exporting to ONNX

This notebook improves the Week-1 detector with Week-2 techniques, compares the
runs against the Week-1 baseline, picks a best model, exports it to ONNX
(NMS-free end-to-end head), and sanity-checks that the ONNX boxes match the
PyTorch boxes. The chosen `best.onnx` is what Part B containerises.

Run this on the same machine / working directory as Week 1 — it expects
`data.yaml` and `DATA_CLEAN/{train,val,test}.txt` to already exist.

## 1. Recap of the Week-1 result

**Week-1 model:** `yolo26s`, 30 epochs, default Ultralytics augmentation.

| Split | mAP@0.5 | mAP@0.5:0.95 | Precision | Recall |
|---|---|---|---|---|
| Validation | 0.908 | 0.717 | 0.895 | 0.868 |
| **Test (held out)** | **0.9305** | **0.7305** | **0.919** | **0.884** |

**Training diagnosis.** Val loss was *still falling* at epoch 30 and val
mAP@0.5:0.95 peaked on the final epoch — this is **under-training with early
divergence**, not overfitting. The run was capped at 30 epochs by the
RTX 4060 Laptop (8 GB) budget, not by convergence.

**Two weaknesses observed in the failure cases.**

1. **Background-texture false positives.** The model boxed a patch of fur-like
   foliage in the same brown/orange tones as a real cat directly above it —
   it appears to have latched onto *"fur-textured blob in a natural setting"*
   as a cue, firing on background texture.
2. **Small / odd-pose false negatives.** A ground-truth cat occupying only
   ~3.7% of the frame, in an unusual pose (curled / viewed from behind), was
   missed entirely even at confidence 0.05.

## 2. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

DATA = "data.yaml"
assert Path(DATA).exists(), "data.yaml not found — run this from the Week-1 working dir."

# Point this at your Week-1 best.pt (used as the baseline row in the comparison).
# Adjust if your run folder differs; the glob picks the most recent cats_v1* run.
_cands = sorted(Path("runs").rglob("cats_v1*/weights/best.pt"), key=lambda p: p.stat().st_mtime)
WEEK1_BEST = str(_cands[-1]) if _cands else "runs/detect/runs/cats_v1-6/weights/best.pt"
print("Week-1 baseline weights:", WEEK1_BEST)
print("exists:", Path(WEEK1_BEST).exists())

## 3. Techniques chosen (and why)

I apply **five** Week-2 techniques across two runs, each aimed at a specific
problem from Section 1.

| Technique | Targets | Rationale |
|---|---|---|
| **Larger backbone (`s → m`)** | under-fit | Week 1 was still improving; `yolo26m` adds capacity. The obvious move for an under-fit run. |
| **Longer training + cosine LR** (`epochs=80`, `cos_lr=True`) | under-fit | Removes the 30-epoch cap; cosine decay anneals cleanly into a stable minimum. |
| **Stronger augmentation** (`mixup`, `copy_paste`, higher `scale`, `mosaic` + late `close_mosaic`) | both weaknesses | `mixup`/`mosaic` break the "fur-texture = cat" shortcut by forcing context mixing → fewer texture FPs; `copy_paste` + scale-jitter expose more small/partial cats → better recall. YOLO26's built-in ProgLoss + STAL already help small objects, and stronger scale jitter compounds that. |
| **Weight decay** (`0.0008`) | texture FPs | A touch more L2 discourages memorising spurious local texture. |
| **Early stopping** (`patience=20`) | efficiency | Stops wasted epochs once val plateaus, given the laptop budget. |

**Two-run design.** Run 1 throws everything at `yolo26m`. Run 2 applies the
*same recipe* to `yolo26s`, to answer a deployment-relevant question: *does the
gain come from model size, or from the better recipe?* If `s` nearly matches
`m`, we ship the smaller, faster ONNX.

> **Hardware note (RTX 4060 Laptop, 8 GB).** `yolo26m` at 640 px is set to
> `batch=8`. If you hit CUDA OOM, drop to `batch=4` (or `batch=-1` for
> Ultralytics auto-batch). `yolo26s` can stay at `batch=16`.

### 3a. v2 — Run 1: `yolo26m` + full recipe

Larger backbone to fix the under-fit, plus the strong-augmentation +
cosine-LR + weight-decay recipe.

In [ ]:
model_m = YOLO("yolo26m.pt")
res_m = model_m.train(
    data=DATA,
    epochs=80,
    imgsz=640,
    batch=8,            # 8 GB laptop GPU; drop to 4 if CUDA OOM
    device=0,
    seed=42,
    project="runs",
    name="cats_v2_m",
    # --- longer schedule ---
    cos_lr=True,
    patience=20,
    # --- stronger augmentation ---
    mosaic=1.0,
    close_mosaic=15,    # disable mosaic for the last 15 epochs (clean fine-tune)
    mixup=0.15,
    copy_paste=0.10,    # lower to 0.0 if your ultralytics build errors on detect
    scale=0.6,          # more scale jitter -> small-object recall
    degrees=5.0,
    translate=0.1,
    # --- regularisation ---
    weight_decay=0.0008,
)
RUN_M_DIR = Path(model_m.trainer.save_dir)
print("Run 1 dir:", RUN_M_DIR)

### 3b. v2 — Run 2: `yolo26s` + same recipe (shippable small model)

Same recipe, smaller backbone. If this nearly matches Run 1, it's the better
thing to ship (smaller ONNX, faster CPU inference in the container).

In [ ]:
model_s = YOLO("yolo26s.pt")
res_s = model_s.train(
    data=DATA,
    epochs=80,
    imgsz=640,
    batch=16,
    device=0,
    seed=42,
    project="runs",
    name="cats_v2_s",
    cos_lr=True,
    patience=20,
    mosaic=1.0,
    close_mosaic=15,
    mixup=0.15,
    copy_paste=0.10,
    scale=0.6,
    degrees=5.0,
    translate=0.1,
    weight_decay=0.0008,
)
RUN_S_DIR = Path(model_s.trainer.save_dir)
print("Run 2 dir:", RUN_S_DIR)

### 3c. (Optional) Two-stage transfer learning

A sixth technique if you want it: freeze the backbone for a short warm-up so
only the head adapts, then unfreeze with a smaller LR. Enable by setting
`RUN_TWO_STAGE = True`. Adds a `cats_v2_twostage` run you can include in the
comparison below.

In [ ]:
RUN_TWO_STAGE = False
RUN_TS_DIR = None
if RUN_TWO_STAGE:
    m_ts = YOLO("yolo26m.pt")
    # Stage 1: head only (freeze first 10 backbone layers), short warm-up.
    m_ts.train(data=DATA, epochs=10, imgsz=640, batch=8, device=0, seed=42,
               project="runs", name="cats_v2_twostage_s1", freeze=10, lr0=0.01)
    # Stage 2: unfreeze everything, smaller LR, finish the schedule.
    res_ts = m_ts.train(data=DATA, epochs=70, imgsz=640, batch=8, device=0, seed=42,
                        project="runs", name="cats_v2_twostage_s2",
                        cos_lr=True, patience=20, mixup=0.15, scale=0.6,
                        weight_decay=0.0008, lr0=0.004)
    RUN_TS_DIR = Path(m_ts.trainer.save_dir)
    print("Two-stage dir:", RUN_TS_DIR)

## 4. Training / validation curves

Plot the loss and mAP curves for each run from its `results.csv` to confirm the
runs actually converged (val loss flattening, mAP plateauing).

In [ ]:
def plot_curves(run_dir, title):
    df = pd.read_csv(Path(run_dir) / "results.csv")
    df.columns = df.columns.str.strip()
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].plot(df["epoch"], df["train/box_loss"], label="train box")
    ax[0].plot(df["epoch"], df["val/box_loss"], label="val box")
    ax[0].set_title(f"{title} — box loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP@0.5")
    ax[1].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")
    ax[1].set_title(f"{title} — val mAP"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()
    best = df["metrics/mAP50-95(B)"].idxmax()
    print(f"{title}: best epoch {int(df.loc[best,'epoch'])}, "
          f"val mAP@0.5={df.loc[best,'metrics/mAP50(B)']:.4f}, "
          f"mAP@0.5:0.95={df.loc[best,'metrics/mAP50-95(B)']:.4f}")

plot_curves(RUN_M_DIR, "v2 run 1 (yolo26m)")
plot_curves(RUN_S_DIR, "v2 run 2 (yolo26s)")

## 5. Test-set comparison against the Week-1 baseline

Evaluate every candidate on the held-out **test** split (`split="test"`) so the
comparison is apples-to-apples, then assemble the required table.

In [ ]:
def eval_test(weights):
    m = YOLO(str(weights))
    mt = m.val(data=DATA, split="test", verbose=False)
    return dict(map50=mt.box.map50, map=mt.box.map, p=mt.box.mp, r=mt.box.mr)

candidates = [
    ("Week-1 baseline", WEEK1_BEST,                       "yolo26s", "none"),
    ("v2 — run 1",      RUN_M_DIR/"weights"/"best.pt",    "yolo26m", "s→m, cos_lr, strong_aug, wd, early_stop"),
    ("v2 — run 2",      RUN_S_DIR/"weights"/"best.pt",    "yolo26s", "cos_lr, strong_aug, wd, early_stop"),
]
if RUN_TS_DIR is not None:
    candidates.append(("v2 — two-stage", RUN_TS_DIR/"weights"/"best.pt", "yolo26m", "two_stage, cos_lr, strong_aug, wd"))

rows = []
for name, w, backbone, tricks in candidates:
    if not Path(w).exists():
        print("skip (missing):", name, w); continue
    m = eval_test(w)
    rows.append({"Run": name, "Backbone": backbone, "Tricks": tricks,
                 "mAP@0.5": round(m["map50"], 4), "mAP@0.5:0.95": round(m["map"], 4),
                 "P": round(m["p"], 4), "R": round(m["r"], 4)})

comp = pd.DataFrame(rows)
display(comp)

best_row = comp.loc[comp["mAP@0.5:0.95"].idxmax()]
print("\nBest run by test mAP@0.5:0.95:", best_row["Run"], "->", best_row["Backbone"])

### Pick the model to ship

Choose the row with the best test mAP@0.5:0.95 (or the best
accuracy-per-MB trade-off if `s` is close to `m`). Set `BEST_WEIGHTS` below to
that run's `best.pt`. The cells under Section 6 export *that* model.

In [ ]:
# Set this to the winning run's weights, e.g. RUN_M_DIR/"weights"/"best.pt"
BEST_WEIGHTS = RUN_M_DIR / "weights" / "best.pt"
print("Shipping:", BEST_WEIGHTS, "| exists:", Path(BEST_WEIGHTS).exists())

## 6. Export to ONNX (NMS-free end-to-end)

YOLO26's default One-to-One head exports a self-contained model whose ONNX
output is `(1, 300, 6)` — `[x1, y1, x2, y2, score, class]`, already final
detections, no NMS needed. `dynamic=False` fixes the input at 640×640.

In [ ]:
ship = YOLO(str(BEST_WEIGHTS))
onnx_path = ship.export(format="onnx", imgsz=640, opset=17, dynamic=False)
print("Exported:", onnx_path)

In [ ]:
# Confirm the output shape before trusting it (per the assessment's advice).
import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
print("input :", sess.get_inputs()[0].name, sess.get_inputs()[0].shape)
print("output:", sess.get_outputs()[0].name, sess.get_outputs()[0].shape)
# Expect output shape (1, 300, 6) for the e2e head.

## 7. Sanity check — ONNX boxes ≈ PyTorch boxes

Run the *same* images through (a) the original PyTorch model via
`ultralytics.predict` and (b) the ONNX model via `onnxruntime` using the exact
letterbox/decoding the container uses. Match boxes by IoU and report the
largest corner difference in pixels — it should be sub-pixel.

In [ ]:
import numpy as np
from PIL import Image

# --- minimal copy of the container's CatDetector logic (e2e head) ---
class ONNXDetector:
    def __init__(self, onnx_path, imgsz=640, conf=0.25):
        self.s = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        self.imgsz, self.conf = imgsz, conf
        self.inp = self.s.get_inputs()[0].name
    def _lb(self, img):
        w0, h0 = img.size
        r = min(self.imgsz/h0, self.imgsz/w0)
        nw, nh = round(w0*r), round(h0*r)
        dw, dh = (self.imgsz-nw)/2, (self.imgsz-nh)/2
        if (nw, nh) != (w0, h0): img = img.resize((nw, nh), Image.BILINEAR)
        left, top = round(dw-0.1), round(dh-0.1)
        canvas = Image.new("RGB", (self.imgsz, self.imgsz), (114,114,114))
        canvas.paste(img, (left, top))
        x = (np.asarray(canvas, np.float32)/255.0).transpose(2,0,1)[None]
        return np.ascontiguousarray(x), r, (left, top)
    def predict(self, path):
        img = Image.open(path).convert("RGB"); W, H = img.size
        x, r, (px, py) = self._lb(img)
        out = self.s.run(None, {self.inp: x})[0][0]   # (300,6)
        boxes = []
        for x1,y1,x2,y2,sc,cl in out:
            if sc < self.conf: continue
            b = [max(0,min(W,(x1-px)/r)), max(0,min(H,(y1-py)/r)),
                 max(0,min(W,(x2-px)/r)), max(0,min(H,(y2-py)/r)), float(sc)]
            boxes.append(b)
        return sorted(boxes, key=lambda b: -b[4])

def iou(a, b):
    ix1,iy1 = max(a[0],b[0]), max(a[1],b[1])
    ix2,iy2 = min(a[2],b[2]), min(a[3],b[3])
    iw,ih = max(0,ix2-ix1), max(0,iy2-iy1); inter = iw*ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/(ua+1e-9)

with open("DATA_CLEAN/test.txt") as f:
    test_imgs = [l.strip() for l in f][:8]

det = ONNXDetector(onnx_path, conf=0.25)
pt = YOLO(str(BEST_WEIGHTS))
max_diff = 0.0
for p in test_imgs:
    pr = pt.predict(source=p, imgsz=640, conf=0.25, device=0, verbose=False)[0]
    pt_boxes = sorted([list(b.xyxy[0].cpu().numpy())+[float(b.conf[0])] for b in pr.boxes],
                      key=lambda b:-b[4])
    on_boxes = det.predict(p)
    for ob in on_boxes:
        if not pt_boxes: break
        j = max(range(len(pt_boxes)), key=lambda k: iou(ob, pt_boxes[k]))
        if iou(ob, pt_boxes[j]) > 0.5:
            max_diff = max(max_diff, max(abs(ob[i]-pt_boxes[j][i]) for i in range(4)))
    print(f"{Path(p).name}: PyTorch={len(pt_boxes)} boxes, ONNX={len(on_boxes)} boxes")
print(f"\nMax matched corner difference: {max_diff:.3f} px  (expect < ~1 px)")

## 8. Ship it

Copy the exported model into the Part B container folder as `best.onnx`:

```bash
cp runs/cats_v2_m/weights/best.onnx container/models/best.onnx
```

Then update `container/STUDENT.json` so `model.variant`, `epochs_total`, and
`tricks` match the run you actually shipped, and continue with Part B
(Dockerfile build, local test, push).